# Vision-Based Autonomous Navigation for UGV
## Multi-Terrain Demo — Creek, Village, Trail

> **Run-All** → produces 3 individual demo videos + 1 montage, then launches Streamlit UI.
>
> **Terrains:** Creek (rock bed) | Village (buildings, roads) | Trail (forest path)

In [ ]:
import os, sys, subprocess, pathlib, warnings, importlib
warnings.filterwarnings('ignore')

# ---- Auto-detect environment (Colab vs local) ----
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    print('Running locally')

# ---- Set REPO_DIR based on environment ----
if IN_COLAB:
    REPO_DIR = pathlib.Path('/content/vision-ugv-nav-rocky')
    if not REPO_DIR.exists():
        print('Cloning repository...')
        subprocess.run(['git', 'clone', 'https://github.com/zerowraith/vision-ugv-nav-rocky.git', str(REPO_DIR)], check=True)
    else:
        print('Repository already present.')
else:
    REPO_DIR = pathlib.Path.cwd()
    print(f'Using local repo at {REPO_DIR}')

# ---- Require GPU ----
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU REQUIRED but not detected. '
        'In Colab: Runtime -> Change runtime type -> GPU (T4). '
        'Locally: ensure nvidia-smi works and CUDA toolkit is installed.'
    )
GPU_NAME = torch.cuda.get_device_name(0)
_props = torch.cuda.get_device_properties(0)
GPU_MEM = getattr(_props, 'total_memory', getattr(_props, 'total_mem', 0)) / 1e9
print(f'GPU: {GPU_NAME} ({GPU_MEM:.1f} GB)')
print(f'CUDA: {torch.version.cuda}, cuDNN: {torch.backends.cudnn.version()}')

# ---- Install system build deps (Colab only) ----
if IN_COLAB:
    !apt-get update -qq && apt-get install -y -qq cmake build-essential libopencv-dev wget unzip ffmpeg 2>&1 | tail -5
    !ffmpeg -encoders 2>&1 | grep libx264 || echo 'WARNING: libx264 not found in ffmpeg'

# ---- Install Python packages ----
print('Installing core dependencies...')
if IN_COLAB:
    !pip install -q "numpy<2.3" "protobuf<5.0.0" "pyngrok" "streamlit" "scipy" "tqdm" "torch" "torchvision" "Pillow==10.4.0" "gdown"
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                     'numpy<2.3', 'protobuf<5.0.0', 'pyngrok', 'streamlit',
                     'scipy', 'tqdm', 'torch', 'torchvision', 'Pillow==10.4.0', 'gdown'],
                    check=False)

import site
importlib.reload(site)

# ---- Add paths ----
os.chdir(REPO_DIR)
for p in [str(REPO_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ---- Verify imports ----
import torchvision, cv2, numpy as np
print(f'torchvision {torchvision.__version__}, opencv {cv2.__version__}')
print('Setup complete — GPU ready.')

In [ ]:
# --------------------
# 1  Download RUGD dataset — extract Creek, Village, Trail
# --------------------
import os, json, subprocess, time
from pathlib import Path
import numpy as np
import cv2

RUGD_BASE = REPO_DIR / 'data' / 'rugd'

# Scene config: folder_name -> (output_dir_name, display_label)
SCENES = {
    'creek':    ('scene_03', 'Creek'),
    'village':  ('village',  'Village'),
    'trail-7':  ('trail_7',  'Trail'),
}

# Camera intrinsics (same for all RUGD scenes)
META = {
    'width': 640, 'height': 480, 'fps': 10,
    'K': [381.362, 0.0, 320.5,
          0.0, 381.362, 240.5,
          0.0, 0.0, 1.0],
    'dist': [0.0, 0.0, 0.0, 0.0, 0.0],
    'camera_height': 1.2,
    'pitch_deg': 0.0,
}

# Check if all scenes already downloaded
all_ready = True
for zip_folder, (out_dir, _) in SCENES.items():
    rgb = RUGD_BASE / out_dir / 'rgb'
    n = len(list(rgb.glob('*.png'))) if rgb.exists() else 0
    if n == 0:
        all_ready = False
        break
    print(f'  {out_dir}: {n} frames')

if all_ready:
    print('All scenes already downloaded.')
else:
    print('Downloading RUGD dataset (~5.3 GB)...')
    ZIP_URL = 'http://rugd.vision/data/RUGD_frames-with-annotations.zip'
    ZIP_PATH = '/tmp/rugd_frames.zip'

    if not Path(ZIP_PATH).exists():
        t0 = time.time()
        try:
            subprocess.run(['wget', '-q', '--show-progress', '-O', ZIP_PATH, ZIP_URL], check=True)
        except (subprocess.CalledProcessError, FileNotFoundError):
            import urllib.request
            def _progress(bn, bs, ts):
                pct = bn * bs / ts * 100 if ts > 0 else 0
                print(f'\r  Downloading: {pct:.1f}%', end='', flush=True)
            urllib.request.urlretrieve(ZIP_URL, ZIP_PATH, reporthook=_progress)
        print(f'\nDownload complete ({time.time()-t0:.0f}s).')

    # Extract all 3 scenes
    import zipfile
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for zip_folder, (out_dir, label) in SCENES.items():
            rgb_dir = RUGD_BASE / out_dir / 'rgb'
            rgb_dir.mkdir(parents=True, exist_ok=True)
            if len(list(rgb_dir.glob('*.png'))) > 0:
                print(f'  {label}: already extracted.')
                continue
            members = [m for m in zf.namelist()
                       if f'/{zip_folder}/' in m.lower() and m.lower().endswith('.png')]
            print(f'  {label}: extracting {len(members)} frames...')
            for j, member in enumerate(sorted(members)):
                data = zf.read(member)
                out_path = rgb_dir / f'frame_{j:04d}.png'
                with open(out_path, 'wb') as f:
                    f.write(data)
                if (j + 1) % 100 == 0:
                    print(f'    {j+1}/{len(members)}')
            print(f'    Done: {len(members)} frames')

    os.remove(ZIP_PATH)

# Write meta.json for each scene
for _, (out_dir, _) in SCENES.items():
    scene_dir = RUGD_BASE / out_dir
    scene_dir.mkdir(parents=True, exist_ok=True)
    with open(scene_dir / 'meta.json', 'w') as f:
        json.dump(META, f, indent=2)

print('\nDataset ready.')
for _, (out_dir, label) in SCENES.items():
    n = len(list((RUGD_BASE / out_dir / 'rgb').glob('*.png')))
    print(f'  {label:10s}: {n} frames')

In [ ]:
# --------------------
# 2  Load segmentation model + define reusable pipeline
# --------------------
import os, json, time
import cv2
import numpy as np
import torch
import torchvision
from pathlib import Path
from tqdm.auto import tqdm

# ---- Load DeepLabV3 ONCE (shared across all scenes) ----
print('Loading DeepLabV3-ResNet50 on GPU...')
seg_model = torchvision.models.segmentation.deeplabv3_resnet50(
    weights='DeepLabV3_ResNet50_Weights.DEFAULT'
).cuda().eval()
print(f'Model loaded — {sum(p.numel() for p in seg_model.parameters())/1e6:.1f}M params')

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

def segment_frame(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (640, 360))
    tensor = torch.from_numpy(resized).permute(2, 0, 1).unsqueeze(0).float().cuda() / 255.0
    tensor = (tensor - IMAGENET_MEAN) / IMAGENET_STD
    with torch.no_grad():
        out = seg_model(tensor)['out']
    raw = out.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
    nav = np.ones_like(raw, dtype=np.uint8)  # default obstacle
    TRAV = {0, 7, 10, 11, 12, 13, 14}
    SKY = {17}
    for c in TRAV:
        nav[raw == c] = 0
    for c in SKY:
        nav[raw == c] = 2
    return nav

from src.mapping.costmap_builder import build_costmap
from src.planning.astar_planner import astar
from src.planning.pure_pursuit import pure_pursuit_step
from src.mapping.viz import draw_frame
from src.slam.visual_odometry import SimpleVisualOdometry
from src.sim.trajectory import simulate_trajectory, SCENE_TRAJECTORIES

SCENE_SCALES = {
    "scene_03": 0.5,   # Creek — moderate speed
    "village":  0.6,   # Village — slightly faster
    "trail_7":  0.4,   # Trail — slower, rougher terrain
}

def run_pipeline(scene_name, scene_dir, scene_label):
    """Full pipeline for one terrain scenario."""
    print(f'\n{"="*60}')
    print(f'  SCENARIO: {scene_label}')
    print(f'{"="*60}')

    out_dir = REPO_DIR / 'data' / 'rugd' / scene_dir
    rgb_dir = out_dir / 'rgb'
    frames = sorted(rgb_dir.glob('*.png'))
    if len(frames) == 0:
        print(f'  ERROR: No frames at {rgb_dir}')
        return None

    # Load camera intrinsics
    with open(out_dir / 'meta.json') as f:
        meta = json.load(f)
    K = np.array(meta['K']).reshape(3, 3)
    CAM_H = meta['camera_height']
    PITCH = np.deg2rad(meta['pitch_deg'])

    # ---- 1. Segmentation ----
    masks_file = out_dir / 'masks.npy'
    if masks_file.exists():
        masks = np.load(masks_file)
        print(f'  Segmentation: loaded cached {masks.shape}')
    else:
        t0 = time.time()
        masks = []
        for fp in tqdm(frames, desc=f'  {scene_label} Segmentation'):
            masks.append(segment_frame(cv2.imread(str(fp))))
        masks = np.stack(masks)
        np.save(masks_file, masks)
        print(f'  Segmentation: {masks.shape} in {time.time()-t0:.1f}s')

    # ---- 2. Visual Odometry (ORB + Essential Matrix) ----
    poses_file = out_dir / 'poses.txt'
    scale = SCENE_SCALES.get(scene_dir, 0.5)
    vo = SimpleVisualOdometry(K, scale=scale)
    for fp in tqdm(frames, desc=f'  {scene_label} VO'):
        frame = cv2.imread(str(fp))
        vo.process_frame(frame)
    poses = vo.get_poses()

    # Check if VO produced meaningful movement
    total_dist = np.sqrt(np.sum(np.diff(poses[:, :2], axis=0)**2))
    if total_dist < 2.0:
        print(f'  WARNING: VO moved only {total_dist:.1f}m — falling back to simulation')
        poses = simulate_trajectory(len(frames), **SCENE_TRAJECTORIES.get(scene_dir, {}))
    else:
        print(f'  VO trajectory: {poses.shape}, {total_dist:.1f}m total distance')

    np.savetxt(poses_file, poses)
    valid = np.ones(len(poses), dtype=bool)

    # ---- 3. Costmap ----
    costmap, origin = build_costmap(
        masks=masks, poses=poses, K=K,
        cam_height=CAM_H, pitch=PITCH,
        resolution=0.05, grid_size_m=80.0, inflate_radius=0.3
    )
    np.save(out_dir / 'costmap.npy', costmap)
    np.save(out_dir / 'origin.npy', origin)
    print(f'  Costmap: {costmap.shape}')

    # ---- 4. Planning ----
    start_xy = poses[0][:2]
    goal_angle = np.arctan2(poses[-1][1] - poses[0][1],
                             poses[-1][0] - poses[0][0])
    goal_xy = start_xy + 15.0 * np.array([np.cos(goal_angle), np.sin(goal_angle)])
    waypoints = astar(costmap, origin, start_xy, goal_xy, resolution=0.05)
    cmd_vel = []
    for p in poses:
        v, w = pure_pursuit_step(p, waypoints, lookahead=1.5)
        cmd_vel.append([v, w])
    cmd_vel = np.array(cmd_vel)
    np.save(out_dir / 'cmd_vel.npy', cmd_vel)
    np.save(out_dir / 'waypoints.npy', np.array(waypoints))
    print(f'  Planning: {len(waypoints)} waypoints')

    # ---- 5. Render video ----
    TARGET_H, TARGET_W = 360, 640
    video_path = out_dir / 'demo.mp4'
    proc = subprocess.Popen(
        ['ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
         '-s', f'{TARGET_W}x{TARGET_H}', '-pix_fmt', 'bgr24', '-r', '10',
         '-i', '-', '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
         '-crf', '20', '-movflags', '+faststart', str(video_path)],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    # Title card
    title = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8)
    for row in range(TARGET_H):
        t = row / TARGET_H
        title[row] = (int(20+30*t), int(10+15*t), int(40+20*t))
    cv2.putText(title, f'{scene_label} Navigation',
                (TARGET_W//2 - 140, 150), cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (0, 212, 255), 2, cv2.LINE_AA)
    cv2.putText(title, 'DeepLabV3  |  Visual Odometry  |  A* + Pure Pursuit',
                (TARGET_W//2 - 200, 200), cv2.FONT_HERSHEY_SIMPLEX,
                0.42, (160, 160, 160), 1, cv2.LINE_AA)
    for fi in range(20):
        alpha = min(fi / 10.0, 1.0)
        proc.stdin.write((title * alpha).astype(np.uint8).tobytes())

    # Main frames
    past = []
    for idx in tqdm(range(len(frames)), desc=f'  {scene_label} Rendering'):
        frame = cv2.resize(cv2.imread(str(frames[idx])), (TARGET_W, TARGET_H))
        past.append(poses[idx])
        heading = 0.0
        if len(past) >= 2:
            dx = past[-1][0] - past[-2][0]
            dy = past[-1][1] - past[-2][1]
            heading = np.arctan2(dx, dy)
        vis = draw_frame(
            frame, masks[idx], poses[idx], waypoints, costmap, origin,
            cmd=cmd_vel[idx], resolution=0.05,
            frame_idx=idx, total_frames=len(frames),
            inference_ms=0.0, past_poses=np.array(past), heading=heading
        )
        proc.stdin.write(vis.tobytes())

    proc.stdin.close()
    _, stderr = proc.communicate()
    if proc.returncode != 0:
        print(f'  ffmpeg error: {stderr.decode()[-200:]}')
        return None

    sz = os.path.getsize(video_path) / 1024
    print(f'  Video: {video_path.name} ({sz:.0f} KB)')
    return out_dir

In [ ]:
# --------------------
# 3  Run all 3 terrain scenarios
# --------------------
import time

RESULTS = {}
t_total = time.time()

for zip_folder, (out_dir, label) in SCENES.items():
    t0 = time.time()
    result = run_pipeline(zip_folder, out_dir, label)
    elapsed = time.time() - t0
    if result:
        RESULTS[label] = result
        print(f'  Completed in {elapsed:.1f}s\n')
    else:
        print(f'  FAILED ({elapsed:.1f}s)\n')

print(f'\nAll scenarios: {time.time()-t_total:.1f}s total')
print(f'Successful: {list(RESULTS.keys())}')

In [ ]:
# --------------------
# 4  Render montage video (side-by-side all terrains)
# --------------------
import subprocess, os
from pathlib import Path
import cv2
import numpy as np
from tqdm.auto import tqdm

# Find all individual demo videos
video_files = []
for label, out_dir in RESULTS.items():
    mp4 = out_dir / 'demo.mp4'
    if mp4.exists():
        video_files.append((label, mp4))

if len(video_files) < 2:
    print('Need at least 2 videos for montage. Skipping.')
else:
    print(f'Creating montage from {len(video_files)} videos...')

    # Open all videos
    caps = []
    for label, vf in video_files:
        cap = cv2.VideoCapture(str(vf))
        caps.append((label, cap))
        print(f'  {label}: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames, '
              f'{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')

    # Find shortest video length
    min_frames = min(int(c.get(cv2.CAP_PROP_FRAME_COUNT)) for _, c in caps)

    # Montage layout: side by side (640 * N) x 360
    CELL_W, CELL_H = 640, 360
    LABEL_H = 30
    MONTAGE_W = CELL_W * len(caps)
    MONTAGE_H = CELL_H + LABEL_H

    montage_path = REPO_DIR / 'demo_montage.mp4'
    proc = subprocess.Popen(
        ['ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
         '-s', f'{MONTAGE_W}x{MONTAGE_H}', '-pix_fmt', 'bgr24', '-r', '10',
         '-i', '-', '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
         '-crf', '20', '-movflags', '+faststart', str(montage_path)],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    for fi in tqdm(range(min_frames), desc='Montage'):
        canvas = np.zeros((MONTAGE_H, MONTAGE_W, 3), dtype=np.uint8)
        for col, (label, cap) in enumerate(caps):
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (CELL_W, CELL_H))
            else:
                frame = np.zeros((CELL_H, CELL_W, 3), dtype=np.uint8)
            x0 = col * CELL_W
            canvas[LABEL_H:LABEL_H+CELL_H, x0:x0+CELL_W] = frame
            # Label
            cv2.putText(canvas, label, (x0 + 10, 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 212, 255), 2, cv2.LINE_AA)
            # Divider line
            if col > 0:
                cv2.line(canvas, (x0, 0), (x0, MONTAGE_H), (100, 100, 100), 2)

        proc.stdin.write(canvas.tobytes())

    proc.stdin.close()
    _, stderr = proc.communicate()
    for _, cap in caps:
        cap.release()

    if proc.returncode == 0:
        sz = os.path.getsize(montage_path) / 1024
        print(f'Montage: {montage_path.name} ({sz:.0f} KB, {min_frames} frames)')
    else:
        print(f'Montage ffmpeg error: {stderr.decode()[-200:]}')

In [ ]:
# --------------------
# 5  Launch Streamlit UI (with scene selector)
# --------------------
import subprocess, sys, time, threading, os

UI_SCRIPT = str(REPO_DIR / 'src' / 'ui' / 'streamlit_app.py')

if IN_COLAB:
    try:
        from pyngrok import ngrok
        from google.colab import userdata
        try:
            authtoken = userdata.get('NGROK_AUTHTOKEN')
            ngrok.set_auth_token(authtoken)
        except Exception:
            print('NGROK_AUTHTOKEN not found - trying without auth')

        os.system('fuser -k 8501/tcp 2>/dev/null || true')

        def run_streamlit():
            subprocess.run([
                sys.executable, '-m', 'streamlit', 'run', UI_SCRIPT,
                '--server.port=8501', '--server.address=0.0.0.0',
                '--server.headless=true'
            ])

        t = threading.Thread(target=run_streamlit, daemon=True)
        t.start()
        time.sleep(8)

        for tunnel in ngrok.get_tunnels():
            ngrok.disconnect(tunnel.public_url)
        public_url = ngrok.connect(8501, bind_tls=True).public_url
        print(f'\nStreamlit UI: {public_url}')
        print('Open the link above. If "Connection Refused", wait 5s and refresh.')
    except Exception as e:
        print(f'Failed to start ngrok tunnel: {e}')
        print('Run manually: python -m streamlit run', UI_SCRIPT)
else:
    print('Streamlit UI available. To launch:')
    print(f'  python -m streamlit run {UI_SCRIPT}')
    launch = input('Launch Streamlit now? [y/N]: ').strip().lower()
    if launch == 'y':
        subprocess.Popen([
            sys.executable, '-m', 'streamlit', 'run', UI_SCRIPT,
            '--server.port=8501', '--server.headless=true'
        ])
        time.sleep(3)
        print('Streamlit running at http://localhost:8501')